<a href="https://colab.research.google.com/github/jithender210/Flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jithender210/Flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


task type :binary classification



why: evaluating input data and assing instance to outcome . it helps in enable a non linear decision boundary

In [4]:
import pandas as pd

# Load your lane slice / dataset
df = pd.read_csv('content_refresh_anonymized.csv')

# Print DataFrame columns to help identify the target column
print("Available columns:", df.columns.tolist())

# Verify target balance (replace 'target' with your label column name)
print(df['trend_direction'].value_counts(normalize=True))

print("Verified task type: Binary Classification")

Available columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
trend_direction
down      0.542067
stable    0.198733
up        0.146267
new       0.074533
flat      0.038400
Name: proportion, dtype: float64
Verified task type: Binary Classification


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target / Prediction: A binary label (1 or 0) representing whether a user successfully completes a key conversion or positive event.



In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check for missing values in target column and confirm data sanity
print("Target missing count:", df['trend_direction'].isna().sum())
print("Unique target labels:", df['trend_direction'].unique())

# Example check for target construction
print(df.groupby('trend_direction').size())

Target missing count: 0
Unique target labels: ['down' 'stable' 'new' 'up' 'flat']
trend_direction
down      16262
flat       1152
new        2236
stable     5962
up         4388
dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Metric: PR-AUC (Precision-Recall Area Under Curve) or F1-Score.
What makes GOOD:An F1-Score >0.85 with minimum precision of 0.85


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Evaluate baseline dummy classifier for metric ground truth
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

# Example check against a naive baseline
y_true = df['trend_direction']

# Identify the majority class from y_true
majority_class = y_true.value_counts().idxmax()

y_pred_baseline = [majority_class] * len(y_true)  # Predict majority class as a string
print(classification_report(y_true, y_pred_baseline, zero_division=0))

              precision    recall  f1-score   support

        down       0.54      1.00      0.70     16262
        flat       0.00      0.00      0.00      1152
         new       0.00      0.00      0.00      2236
      stable       0.00      0.00      0.00      5962
          up       0.00      0.00      0.00      4388

    accuracy                           0.54     30000
   macro avg       0.11      0.20      0.14     30000
weighted avg       0.29      0.54      0.38     30000



## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*


Unit of Analysis: One row = One specific interaction instance or individual profile query (e.g., one user-recommendation pair).

key columns :
   Identifiers: user_id, item_id
   Features: skill_overlap_ratio, profile_completeness_score
   Target: target (0 or 1)

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Dataset Shape:", df.shape)
df[['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier','trend_direction']].head()

Dataset Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Manual if else can ot efficiently balance high dimensional attributes

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Measure feature correlations to show non-trivial relationships

# The 'trend_direction' column is categorical (string) and not included in numeric correlation by default.
# To calculate correlation with a numeric target for binary classification, we first need to convert it.
# Assuming for a binary classification task, we want to distinguish 'up' trend from others.
# You might need to adjust this mapping based on your specific binary target definition.

# Create a binary numeric target column (e.g., 1 for 'up', 0 for others)
df['binary_trend_target'] = (df['trend_direction'] == 'up').astype(int)

correlation = df.corr(numeric_only=True)['binary_trend_target'].sort_values(ascending=False)
print("Feature Correlations with Binary Trend Target:\n", correlation)

Feature Correlations with Binary Trend Target:
 binary_trend_target       1.000000
trend_pct                 0.183289
avg_position              0.167844
content_age_days          0.100780
age_tier_order            0.100095
impressions_last_30d      0.054618
days_with_impressions     0.038356
competition               0.024083
engagement_rate           0.022666
clicks_last_30d           0.019912
search_volume             0.014328
cpc                       0.012186
ctr                       0.006917
sessions_last_30d         0.002925
engaged_sessions_90d      0.002908
ai_traffic_pct           -0.001027
ai_sessions_90d          -0.007426
clicks_90d               -0.011870
impressions_90d          -0.011905
days_with_sessions       -0.017117
clicks_prev_30d          -0.019626
pageviews_90d            -0.020553
scroll_rate              -0.022059
sessions_prev_30d        -0.022832
scroll_events_90d        -0.024027
sessions_90d             -0.024514
users_90d                -0.025615
days_si

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.